In [ ]:
import torch 
import numpy as np 


We will decompose a W matrix, find out the rank of the matrix, find out B & A using the rank and the decomposed 
matrices of W. Finally we will simply do a linear transformation y = W.x + bias to show W and (B, A) are interchangeable.

In [15]:
torch.manual_seed(122)
d, k = 10,10 

W_rank = 2 
W = torch.randn((d, W_rank)) @ torch.randn(W_rank, k)
print(W)
print(W.shape)

tensor([[-2.0001e+00,  2.3976e-01, -5.5540e-01,  1.5095e+00, -2.6808e+00,
         -2.2373e+00, -1.9837e+00, -5.4537e-01,  2.5382e+00,  2.7962e+00],
        [ 1.6001e-01, -3.4455e-01, -1.3207e+00,  2.0341e+00, -3.5271e+00,
         -2.1811e+00, -1.5149e+00, -9.0647e-01,  2.6517e+00,  1.2144e+00],
        [-7.5354e-01,  2.9089e-01,  6.3219e-01, -7.5951e-01,  1.2962e+00,
          6.1186e-01,  2.8423e-01,  3.8017e-01, -8.0335e-01,  1.6703e-01],
        [-5.8717e+00,  1.0929e+00,  1.4295e-03,  1.8556e+00, -3.3973e+00,
         -3.7467e+00, -3.8229e+00, -4.6525e-01,  4.0388e+00,  6.4897e+00],
        [ 2.0351e-01,  3.3400e-01,  1.5602e+00, -2.5272e+00,  4.3941e+00,
          2.8273e+00,  2.0453e+00,  1.1020e+00, -3.4028e+00, -1.8686e+00],
        [-9.9140e-01,  3.1314e-01,  5.3988e-01, -5.3853e-01,  9.0545e-01,
          3.0038e-01,  1.6122e-02,  2.9703e-01, -4.4658e-01,  5.2723e-01],
        [ 2.1613e+00, -2.0751e-01,  8.1654e-01, -1.9728e+00,  3.4900e+00,
          2.7917e+00,  2.4088e+0

In [17]:
#Lets calculate the rank of this matrix 
W_rank = np.linalg.matrix_rank(W)
print(f"Rank is: {W_rank}")

Rank is: 2


In [29]:
#Decompose the matrix W using SVD into 3 seperate matrices 
U, S, V = torch.svd(W)

print(U.shape)
print(S.shape)
print(V.shape)

torch.Size([10, 10])
torch.Size([10])
torch.Size([10, 10])


In [ ]:
W_recon = U @ torch.diag(S) @ V.t() #reconstructing W just to make sense of SVD. 

In [32]:
W_recon


tensor([[-2.0001e+00,  2.3976e-01, -5.5540e-01,  1.5095e+00, -2.6808e+00,
         -2.2373e+00, -1.9837e+00, -5.4537e-01,  2.5382e+00,  2.7962e+00],
        [ 1.6001e-01, -3.4455e-01, -1.3207e+00,  2.0341e+00, -3.5271e+00,
         -2.1812e+00, -1.5149e+00, -9.0647e-01,  2.6517e+00,  1.2144e+00],
        [-7.5354e-01,  2.9089e-01,  6.3219e-01, -7.5951e-01,  1.2962e+00,
          6.1186e-01,  2.8423e-01,  3.8017e-01, -8.0335e-01,  1.6703e-01],
        [-5.8717e+00,  1.0929e+00,  1.4302e-03,  1.8556e+00, -3.3973e+00,
         -3.7467e+00, -3.8229e+00, -4.6525e-01,  4.0388e+00,  6.4897e+00],
        [ 2.0351e-01,  3.3400e-01,  1.5602e+00, -2.5272e+00,  4.3941e+00,
          2.8273e+00,  2.0453e+00,  1.1020e+00, -3.4028e+00, -1.8686e+00],
        [-9.9140e-01,  3.1314e-01,  5.3988e-01, -5.3853e-01,  9.0545e-01,
          3.0039e-01,  1.6122e-02,  2.9703e-01, -4.4658e-01,  5.2723e-01],
        [ 2.1613e+00, -2.0751e-01,  8.1654e-01, -1.9728e+00,  3.4900e+00,
          2.7917e+00,  2.4088e+0

In [35]:
U_r  = U[:,:W_rank]
U_r.shape

torch.Size([10, 2])

In [37]:
S_r = torch.diag(S[:W_rank])
S_r.shape

torch.Size([2, 2])

In [38]:
V_r = V[:,:W_rank].t()
V_r.shape

torch.Size([2, 10])

In [ ]:
#now, construct B and A matrices of LoRa 
B = U_r @ S_r 
A = V_r

print(f"Shape of B is: {B.shape}")
print(f"Shape of A is: {A.shape}") 

Shape of B is: torch.Size([10, 2])
Shape of A is: torch.Size([2, 10])


In [40]:
#doing a simple linear transformation (simulating a neural network operation to show W can be replaced with B & A)
bias = torch.randn(d)
x = torch.randn(d)

y = W @ x + bias
y_lora = (B @ A) @ x + bias

print(f"Original y using W:\n", y)
print(f"y using B & A:\n", y_lora)


Original y using W:
 tensor([ 6.7399,  8.5491, -2.2973, 11.5482, -9.4582, -2.1800, -8.2115, -0.1219,
        -7.4595,  0.5474])
y using B & A:
 tensor([ 6.7399,  8.5491, -2.2973, 11.5482, -9.4582, -2.1800, -8.2115, -0.1219,
        -7.4595,  0.5474])


In [ ]:
#lets see the total parameters that we used for both cases 

print(f"Parameters for W: {W.nelement()}")
print(f"Parameters for B & A : {B.nelement() + A.nelement()}")

Parameters for W: 100
Parameters for B & A : 40
